In [2]:
import numpy as np
import networkx as nx
import cvxpy as cp

# =========================
# 1. Generate graph
# =========================
def generate_graph(n=50, p=0.3, seed=0):
    return nx.erdos_renyi_graph(n, p, seed=seed)

# =========================
# 2. Solve SDP relaxation
# =========================
def solve_sdp(G):
    n = G.number_of_nodes()
    X = cp.Variable((n, n), symmetric=True)
    constraints = [X >> 0, cp.diag(X) == 1]

    # OPTIMIZATION: Use the Graph Laplacian for a much faster CVXPY compilation
    L = nx.laplacian_matrix(G).toarray()
    obj = cp.trace(L @ X) / 4

    prob = cp.Problem(cp.Maximize(obj), constraints)
    
    # CLARABEL is generally faster/more accurate for SDPs if installed, 
    # but SCS remains a fine fallback.
    prob.solve(solver=cp.SCS)

    return X.value, prob.value

# =========================
# 3. Factor SDP solution
# =========================
def factorize_sdp(X):
    # Your approach here is excellent for handling numerical inaccuracies
    eigvals, eigvecs = np.linalg.eigh(X)
    eigvals = np.clip(eigvals, 0, None)
    return eigvecs @ np.diag(np.sqrt(eigvals))

# =========================
# 4. Hyperplane rounding
# =========================
def gw_rounding(G, V, num_samples=1000): # Increased default samples due to speed
    n, d = V.shape
    edges = np.array(list(G.edges()))

    # OPTIMIZATION: Fully vectorized random hyperplanes
    # Generate all random vectors at once: shape (d, num_samples)
    R = np.random.randn(d, num_samples)
    R /= np.linalg.norm(R, axis=0)

    # Compute signs for all samples at once: shape (n, num_samples)
    signs = np.sign(V @ R)
    signs[signs == 0] = 1  # Handle edge case where projection is exactly 0

    # Vectorized cut calculation
    # signs[edges[:, 0]] gives an array of shape (|E|, num_samples)
    cuts = np.sum(signs[edges[:, 0]] != signs[edges[:, 1]], axis=0)

    # Find the best cut across all samples
    best_idx = np.argmax(cuts)
    best_cut = cuts[best_idx]
    best_partition = signs[:, best_idx]

    return best_cut, best_partition

# =========================
# 5. Full pipeline
# =========================
def goemans_williamson(G):
    print("Solving SDP...")
    X, sdp_val = solve_sdp(G)

    print("Rounding...")
    V = factorize_sdp(X)
    cut, partition = gw_rounding(G, V, num_samples=1000)

    print("\nResults:")
    print(f"SDP value: {sdp_val:.2f}")
    print(f"Cut value: {cut}")
    print(f"Approx ratio: {cut / sdp_val:.3f}")

    return cut, partition

# =========================
# RUN
# =========================
if __name__ == "__main__":
    G = generate_graph(n=50, p=0.3)
    goemans_williamson(G)

Solving SDP...
Rounding...

Results:
SDP value: 250.24
Cut value: 243
Approx ratio: 0.971
